# Sales & Revenue Analytics — Exploratory Data Analysis

This notebook performs exploratory data analysis on the corporate sales dataset including:
- Dataset overview
- Missing value analysis
- Statistical summary
- Revenue distribution
- Profit analysis
- Customer analysis
- Correlation analysis

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_processing import preprocess_pipeline

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

df, kpis = preprocess_pipeline(PROJECT_ROOT / "data" / "sales_data.csv")
print(f"Dataset shape: {df.shape}")

## 1. Dataset Overview

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
print("Unique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()}")

## 2. Missing Value Analysis

In [ ]:
raw_df = pd.read_csv(PROJECT_ROOT / "data" / "sales_data.csv")
missing = raw_df.isnull().sum()
missing_pct = (missing / len(raw_df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df[missing_df["Missing Count"] > 0]

In [ ]:
if missing.sum() > 0:
    sns.barplot(x=missing.index, y=missing.values)
    plt.title("Missing Values by Column (Raw Data)")
    plt.xticks(rotation=45)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values after preprocessing.")

## 3. Statistical Summary

In [ ]:
df.describe().T

In [ ]:
print(f"Total Revenue:   ${kpis['total_revenue']:,.2f}")
print(f"Total Profit:      ${kpis['total_profit']:,.2f}")
print(f"Total Orders:      {kpis['total_orders']:,}")
print(f"Total Customers:   {kpis['total_customers']:,}")
print(f"Average Order Value: ${kpis['average_order_value']:,.2f}")

## 4. Revenue Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Revenue"], bins=50, kde=True, ax=axes[0], color="#2563eb")
axes[0].set_title("Revenue Distribution")
axes[0].set_xlabel("Revenue ($)")

monthly = df.groupby("YearMonth")["Revenue"].sum()
monthly.plot(kind="line", ax=axes[1], marker="o", color="#2563eb")
axes[1].set_title("Monthly Revenue Trend")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Revenue ($)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
region_revenue = df.groupby("Region")["Revenue"].sum().sort_values(ascending=False)
sns.barplot(x=region_revenue.values, y=region_revenue.index, palette="Blues_r")
plt.title("Revenue by Region")
plt.xlabel("Revenue ($)")
plt.tight_layout()
plt.show()

## 5. Profit Analysis

In [ ]:
df["Profit_Margin"] = (df["Profit"] / df["Revenue"] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Profit"], bins=50, kde=True, ax=axes[0], color="#16a34a")
axes[0].set_title("Profit Distribution")
axes[0].set_xlabel("Profit ($)")

category_profit = df.groupby("Product_Category")["Profit"].sum().sort_values()
category_profit.plot(kind="barh", ax=axes[1], color="#16a34a")
axes[1].set_title("Profit by Product Category")
axes[1].set_xlabel("Profit ($)")

plt.tight_layout()
plt.show()

In [ ]:
top_products = kpis["top_selling_products"].head(10)
sns.barplot(data=top_products, x="Revenue", y="Product_Name", palette="Teal")
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue ($)")
plt.tight_layout()
plt.show()

## 6. Customer Analysis

In [ ]:
customer_stats = (
    df.groupby("Customer_ID")
    .agg(
        Total_Revenue=("Revenue", "sum"),
        Order_Count=("Order_ID", "nunique"),
        Avg_Order_Value=("Revenue", "mean"),
    )
    .sort_values("Total_Revenue", ascending=False)
)

customer_stats.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_customers = customer_stats.head(15)
sns.barplot(
    data=top_customers.reset_index(),
    x="Total_Revenue",
    y="Customer_ID",
    ax=axes[0],
    palette="Purples",
)
axes[0].set_title("Top 15 Customers by Revenue")
axes[0].set_xlabel("Total Revenue ($)")

channel_perf = df.groupby("Sales_Channel")["Revenue"].sum().sort_values()
channel_perf.plot(kind="barh", ax=axes[1], color="#7c3aed")
axes[1].set_title("Revenue by Sales Channel")
axes[1].set_xlabel("Revenue ($)")

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
numeric_cols = ["Quantity", "Unit_Price", "Discount", "Revenue", "Profit"]
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f",
    linewidths=0.5,
)
plt.title("Correlation Matrix — Numeric Features")
plt.tight_layout()
plt.show()

In [ ]:
print("Key Correlations with Revenue:")
print(correlation_matrix["Revenue"].sort_values(ascending=False))

## Summary

This EDA reveals revenue patterns across regions, categories, and channels. Key findings:
- Revenue and profit are strongly correlated with quantity and unit price
- Regional and category breakdowns highlight top-performing segments
- Customer concentration shows Pareto-like distribution among top spenders
- Monthly trends support time-series forecasting for future revenue planning